In [1]:
import requests
import json
import pandas as pd

url = "https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.FAD/FM/5.0.0/*.*.A?c[TIME_PERIOD]=ge:2009-10-01+le:2026-12-31&attributes=all&detail=full&includeHistory=true"

r = requests.get(url)

data = r.json()


In [2]:
dataset = data["data"]["dataSets"][0]
structure = data["data"]["structures"][0]

countries = structure["dimensions"]["series"][0]["values"]
indicators = structure["dimensions"]["series"][1]["values"]
years = structure["dimensions"]["observation"][0]["values"]

records = []

for series_key, series_data in dataset["series"].items():

    country_idx, indicator_idx, freq_idx = map(
        int,
        series_key.split(":")
    )

    country_code = countries[country_idx]["id"]
    indicator = indicators[indicator_idx]["id"]

    observations = series_data.get("observations", {})

    for obs_idx, obs_value in observations.items():

        year = int(
            years[int(obs_idx)]["value"]
        )

        value = obs_value[0]

        records.append({
            "country_code": country_code,
            "indicator": indicator,
            "year": year,
            "value": float(value) if value is not None else None
        })

df_imf = pd.DataFrame(records)

In [3]:
indicator_map = {
    "G63G_S13_POGDP_PT": "gross_debt_pct_gdp",
    "GNLB_S13_POGDP_PT": "fiscal_balance_pct_gdp"
}

df_imf["indicator"] = df_imf["indicator"].map(indicator_map)

In [4]:
df_imf_panel = df_imf.pivot_table(
    index=["country_code", "year"],
    columns="indicator",
    values="value"
).reset_index()

df_imf_panel.columns.name = None

print(df_imf_panel.head())
print(df_imf_panel.shape)

  country_code  year  fiscal_balance_pct_gdp  gross_debt_pct_gdp
0          ABW  2010               -3.486738           54.662347
1          ABW  2011               -6.428934           59.346711
2          ABW  2012               -8.954925           65.533056
3          ABW  2013               -6.047697           70.031915
4          ABW  2014               -7.458534           77.709161
(3602, 4)


In [5]:
missing_pct = (
    df_imf_panel
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print(missing_pct)

gross_debt_pct_gdp        1.11
fiscal_balance_pct_gdp    0.11
year                      0.00
country_code              0.00
dtype: float64


In [6]:
id_cols = ["country_code", "year"]

vars_cols = [
    c for c in df_imf_panel.columns
    if c not in id_cols
]

df_imf_imputed = df_imf_panel.copy()

df_imf_imputed[vars_cols] = (
    df_imf_imputed
    .groupby("country_code")[vars_cols]
    .transform(
        lambda x: x.fillna(x.median())
    )
)

In [7]:
df_imf_imputed.isna().sum()

country_code               0
year                       0
fiscal_balance_pct_gdp     0
gross_debt_pct_gdp        40
dtype: int64

In [8]:
id_cols = ["country_code", "year"]

vars_cols = [
    c for c in df_imf_imputed.columns
    if c not in id_cols
]

# Identifica países com alguma variável 100% nula
missing_country_var = (
    df_imf_imputed
    .groupby("country_code")[vars_cols]
    .apply(lambda x: x.isna().mean())
)

countries_to_drop = (
    missing_country_var
    .eq(1)
    .any(axis=1)
)

countries_to_drop = countries_to_drop[
    countries_to_drop
].index.tolist()

print(f"Países removidos: {len(countries_to_drop)}")
print(countries_to_drop)

# Remove do painel
df_imf_imputed = df_imf_imputed[
    ~df_imf_imputed["country_code"].isin(countries_to_drop)
].copy()

Países removidos: 3
['LBY', 'SOM', 'YEM']


In [9]:
(
    df_imf_imputed
    .isna()
    .sum()
    .sort_values(ascending=False)
)

country_code              0
year                      0
fiscal_balance_pct_gdp    0
gross_debt_pct_gdp        0
dtype: int64

In [10]:
imf_countries = set(df_imf_imputed["country_code"].unique())
print(f"FMI: {len(imf_countries)} países")

FMI: 211 países


In [11]:
df_imf_imputed.to_csv(
    "data/imf_panel.csv",
    index=False
)
